# Five-Fold Cross-Validation for Vestibular Schwannoma Segmentation

Compare UNet, DynUNet, and optional SegMamba models for patch-based 3D segmentation of vestibular schwannoma on contrast-enhanced T1-weighted (ceT1) MRI.

## Task and data

The dataset used in this study contains 346 contrast-enhanced T1-weighted (ceT1) scans with matching tumour masks. It combines two separate cohorts: 241 cases from the Vestibular-Schwannoma-SEG dataset at Queen Square Radiosurgery Centre in London [[1](https://doi.org/10.1038/s41597-021-01064-w), [2](https://doi.org/10.7937/TCIA.9YTJ-5Q73)], and 105 cases from Elisabeth-TweeSteden Hospital (ETZ) in Tilburg, released for crossMoDA 2022 [[3](https://doi.org/10.5281/zenodo.6504722), [4](https://doi.org/10.1016/j.media.2022.102628)]. CrossMoDA 2022 also contains London cases, but we exclude them to avoid adding the same data twice.

## Cross-validation and models

The fixed `fold` column assigns every case to exactly one of five validation folds (69–70 cases per fold). All models use the same preprocessing, training, and evaluation pipeline.

## 1. Environment setup

In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchio as tio
from IPython.display import display

from fastMONAI.vision_all import (
    MedDataset, MedImage, MedMask, MedPatchDataLoaders, ZNormalization,
    prediction_filename, preprocess_dataset,
)
from fastMONAI.vision_plot import show_segmentation_comparison

start = Path.cwd().resolve()
if (start / "notebooks").is_dir() and (start / "README.md").is_file():
    PROJECT_ROOT = start
elif start.name == "notebooks" and (start.parent / "README.md").is_file():
    PROJECT_ROOT = start.parent
else:
    raise FileNotFoundError("Start Jupyter from vestibular_schwannoma or its notebooks directory")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from workflow.config import ExperimentConfig, make_gpu_augmentation, make_patch_config
from workflow.models import get_training_model_configs
from workflow.results import (
    aggregate_results, build_model_comparison, format_model_comparison,
    select_qualitative_cases,
)
from workflow.training import run_training_sweep


## 2. Run configuration

Cross-validation and optional all-data training are independent. Use one model, one fold, and two epochs for a smoke test.

Notebook 01 requires a CUDA-capable NVIDIA GPU with bfloat16 support.

Patch-loading settings depend on available hardware. More queue workers may improve training throughput, while a larger queue increases RAM usage.

`training_seed` initializes training randomness before each independent run while retaining cuDNN performance optimizations. In an all-data run, every case remains in training and the first case by stable `case_id` order is duplicated only for fastai's validation phase. Its metric is an internal monitor, not held-out evaluation.

Every independently launched job must use a distinct, previously nonexistent `RESULTS_ROOT`. Re-run this configuration cell before starting another sweep. After parallel fold subsets finish (or one is interrupted after completing some folds), use `merge_parallel_folds.py` to validate and combine their `completed_run_ids.json` registries. The merger rejects different dataset, split, preprocessing, model, loss, or training contracts. A partial `completed_run_ids.json` registry cannot be used directly for inference.

In [ ]:
experiment = ExperimentConfig(
    model_keys=("unet", "dynunet", "segmamba"),
    folds=(1, 2, 3, 4, 5),
    run_cross_validation=True,
    train_all_data=False,
    training_seed=42,
    epochs=500,
    batch_size=4,
    learning_rate=1e-3,
    use_tta=True,
    compile_models=True,
    target_spacing=(0.4102, 0.4102, 1.5),
    patch_size=(256, 256, 48),
    queue_num_workers=4,
    queue_length=300,
)

DATA_CSV = "data/ml_dataset.csv"
RESULTS_RUN = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS_ROOT = Path("cv_results") / RESULTS_RUN

torch.backends.cudnn.benchmark = True

print(f"Models: {experiment.model_keys}")
print(f"Cross-validation: {experiment.run_cross_validation} | folds: {experiment.folds}")
print(f"All-data deployment run: {experiment.train_all_data}")
print(
    f"Epochs: {experiment.epochs} | Batch size: {experiment.batch_size} | "
    f"LR: {experiment.learning_rate} | TTA: {experiment.use_tta}"
)
print(f"Target spacing: {experiment.target_spacing} | Patch size: {experiment.patch_size}")
print(f"Results: {RESULTS_ROOT}")


## 3. Dataset and folds

The CSV supplies stable case IDs, image/mask paths, and fold assignments.

In [ ]:
train_df = pd.read_csv(DATA_CSV)
required_columns = {"case_id", "t1_img_path", "t1_seg_path", "fold"}
missing_columns = sorted(required_columns - set(train_df.columns))
if missing_columns:
    raise ValueError(f"Dataset CSV is missing columns: {missing_columns}")
if train_df["case_id"].duplicated().any():
    raise ValueError("Dataset CSV contains duplicate case_id values")

available_folds = sorted(map(int, train_df["fold"].dropna().unique()))
if experiment.run_cross_validation:
    unknown_folds = sorted(set(experiment.folds) - set(available_folds))
    if unknown_folds:
        raise ValueError(f"Unknown folds requested: {unknown_folds}")

print(f"Total cases: {len(train_df)}")
print()
print("Cases per fold (each fold is the validation set when held out):")
print(train_df["fold"].value_counts().sort_index().to_string())
train_df[["case_id", "t1_img_path", "t1_seg_path", "fold"]].head()


## 4. Preprocess once to disk

Preprocessing is fold-independent. `preprocess_dataset()` automatically creates a versioned preprocessing cache and a `preprocessing_manifest.json` file. On later runs, fastMONAI uses the manifest to verify that the source files and preprocessing settings are unchanged before reusing the cache. `PatchConfig(preprocessed=True)` prevents preprocessing from being applied twice during training.

Populate a new preprocessing cache with one process before launching parallel training jobs; concurrent first-time cache creation is not supported.


### Intensity normalization

Foreground Z-normalization excludes zero-valued image background from its statistics.

In [ ]:
pre_patch_tfms = [ZNormalization(masking_method="foreground")]

label_dataset = MedDataset(
    img_list=train_df.t1_seg_path.tolist(),
    dtype=MedMask,
    max_workers=experiment.preprocess_workers,
)
DATASET_VERSION = label_dataset.fingerprint
if DATASET_VERSION is None:
    raise RuntimeError("Could not compute dataset_version from the segmentation masks")

preprocessing_result = preprocess_dataset(
    train_df,
    img_col="t1_img_path",
    mask_col="t1_seg_path",
    output_dir="preprocessed",
    target_spacing=list(experiment.target_spacing),
    apply_reorder=True,
    transforms=pre_patch_tfms,
    max_workers=experiment.preprocess_workers,
    dataset_version=DATASET_VERSION,
)

print(f"dataset_version: {DATASET_VERSION}")
print(f"preprocessing_cache_version: {preprocessing_result.cache_version}")
print(f"cache reused: {preprocessing_result.reused}")
print(f"manifest: {preprocessing_result.manifest_path}")
print("Added columns:", [column for column in train_df if column.endswith("_preprocessed")])
train_df[["t1_img_path", "t1_img_path_preprocessed"]].head()


## 5. Shared pipeline building blocks

All models share label-biased sampling, preprocessing, overlap aggregation, and post-processing:

- `label_probabilities={0: 0.2, 1: 0.8}` favours tumor-centred patches.
- `normalization` records the inference transform while `preprocessed=True` skips it during training.
- `keep_largest_component=False` preserves every predicted candidate region; no component-size filter is applied.
- Held-out evaluation uses TTA and reports the all-component segmentation, including predicted and zero-overlap component counts.

In [ ]:
patch_config = make_patch_config(experiment, pre_patch_tfms)
patch_config


### GPU augmentation

GPU-batched augmentation follows nnU-Net-style spatial and intensity transforms [[5](https://doi.org/10.1007/978-3-031-72114-4_47)].


In [ ]:
gpu_augmentation_preview = make_gpu_augmentation(experiment)
gpu_augmentation_preview


### Sanity check: inspect a training batch

Inspect clean patches before starting the sweep.

In [ ]:
sanity_fold = available_folds[0]
sanity_df = train_df.copy()
sanity_df["is_val"] = sanity_df["fold"] == sanity_fold

sanity_config = make_patch_config(experiment, pre_patch_tfms)
sanity_config.samples_per_volume = 1
sanity_config.queue_length = experiment.batch_size
sanity_config.queue_num_workers = 2

sanity_dls = MedPatchDataLoaders.from_df(
    df=sanity_df,
    img_col="t1_img_path_preprocessed",
    mask_col="t1_seg_path_preprocessed",
    valid_col="is_val",
    patch_config=sanity_config,
    bs=experiment.batch_size,
)
sanity_dls.show_batch(dl_idx=0, max_n=4, overlay=True, anatomical_plane=2)
sanity_dls.close()


## 6. Model definitions

UNet and SegMamba use Dice plus cross-entropy. DynUNet uses deep supervision through `DynUNetDSAdapter`. UNet and DynUNet are compiled; SegMamba uses its custom CUDA backend.

In [ ]:
model_configs = get_training_model_configs(experiment.model_keys)
print("Registered models:", list(model_configs))
for model_config in model_configs.values():
    print(f"  {model_config.key}: {model_config.model_spec['arch_id']}")


## 7. Run the requested training


In [ ]:
training_sweep = run_training_sweep(
    model_configs,
    train_df,
    experiment=experiment,
    patch_config=patch_config,
    preprocessing_manifest=preprocessing_result.manifest_path,
    results_root=RESULTS_ROOT,
)


## 8. Review run failures


In [ ]:
if training_sweep.failures:
    display(pd.DataFrame([failure.__dict__ for failure in training_sweep.failures]))
else:
    print("All requested runs completed without a recorded failure.")


## 9. Training sweep summary


In [ ]:
completed_folds = {
    model_key: sorted(fold_runs)
    for model_key, fold_runs in training_sweep.fold_runs.items()
}
print("Completed folds:", completed_folds)
if training_sweep.all_data_run_ids:
    print("All-data MLflow runs:", training_sweep.all_data_run_ids)
print("Inference run selection:", training_sweep.inference_run_ids_path)
print("Results root:", RESULTS_ROOT)


## 10. Aggregate per-model results


In [ ]:
cv_combined = {}
for model_key in model_configs:
    combined = aggregate_results(
        RESULTS_ROOT / model_key,
        experiment.folds,
        train_df,
    )
    if combined is not None:
        cv_combined[model_key] = combined


In [ ]:
print("Models with complete cross-validation results:", list(cv_combined))


## 11. Cross-model comparison

Compare models with complete requested-fold results.

In [ ]:
comparison_df = build_model_comparison(cv_combined)
if comparison_df.empty:
    print("No per-model results available yet. Run the training sweep first.")
else:
    comparison_path = RESULTS_ROOT / "cv_model_comparison.csv"
    comparison_df.to_csv(comparison_path)
    print(f"Saved cross-model comparison to {comparison_path}")
    print()
    display(format_model_comparison(comparison_df))


## 12. Qualitative check: prediction vs. ground truth

Show the median- and worst-DSC cases from the first completed model.

In [ ]:
picked = next(iter(cv_combined.items()), None)
if picked is None:
    print("No complete cross-validation results are available for qualitative review.")
else:
    model_key, combined = picked
    results_dir = RESULTS_ROOT / model_key
    print(
        f"Model '{model_key}': pooled {len(combined)} cases from folds "
        f"{sorted(combined['fold'].unique().tolist())}."
    )

    for label, row in select_qualitative_cases(combined):
        case_id = row["case_id"]
        subject = train_df[train_df["case_id"] == case_id]
        if subject.empty:
            print(f"[skip] {case_id}: not found in train_df.")
            continue

        image_path = str(subject.iloc[0]["t1_img_path"])
        ground_truth_path = str(subject.iloc[0]["t1_seg_path"])
        fold = int(row["fold"])
        prediction_path = (
            results_dir / f"fold_{fold}" / "predictions" / prediction_filename(image_path)
        )
        if not prediction_path.exists():
            print(f"[skip] {case_id}: prediction not found at {prediction_path}.")
            continue

        image = MedImage.create(image_path)
        ground_truth = MedMask.create(ground_truth_path)
        prediction = MedMask.create(str(prediction_path))
        voxel_size = tio.ScalarImage(image_path).spacing
        dsc = float(row["dsc"])
        print(f"[{label}] {case_id} (fold {fold}): DSC = {dsc:.4f}")
        show_segmentation_comparison(
            image,
            ground_truth,
            prediction,
            metric_value=dsc,
            metric_name="DSC",
            anatomical_plane=2,
            voxel_size=voxel_size,
        )
        plt.show()


## 13. Viewing runs and next steps

MLflow records parameters, metrics, splits, provenance, and model artifacts. Local outputs are:

- `preprocessed/`: versioned preprocessing caches.
- `cv_results/<RESULTS_RUN>/<model>/fold_N/`: metrics and predictions.
- `cv_results/<RESULTS_RUN>/<model>/cv_summary.csv`: one complete model summary.
- `cv_results/<RESULTS_RUN>/cv_model_comparison.csv`: cross-model summary.
- `cv_results/<RESULTS_RUN>/completed_run_ids.json`: atomically updated MLflow runs for completed folds, including before a later interruption.
- `cv_results/<RESULTS_RUN>/inference_run_ids.json`: exact fully completed MLflow runs for notebook 02.

MLflow retains final and selected weights-only checkpoints plus strict-loadable Safetensors artifacts. All-data runs produce only final artifacts.

Notebook 02 reads `inference_run_ids.json`, then downloads and validates one completed all-data model or fold ensemble.

## References

1. Shapey, J., et al. (2021). [Segmentation of vestibular schwannoma from MRI, an open annotated dataset and baseline algorithm](https://doi.org/10.1038/s41597-021-01064-w). *Scientific Data*, 8, 286.
2. Shapey, J., et al. (2021). [Segmentation of Vestibular Schwannoma from Magnetic Resonance Imaging: An Open Annotated Dataset and Baseline Algorithm](https://doi.org/10.7937/TCIA.9YTJ-5Q73) [Data set]. *The Cancer Imaging Archive*.
3. Dorent, R., Kujawa, A., Cornelissen, S., Langenhuizen, P. P. J. H., Shapey, J., & Vercauteren, T. (2022). [Cross-Modality Domain Adaptation Challenge 2022 (crossMoDA)](https://doi.org/10.5281/zenodo.6504722) [Data set]. *Zenodo*.
4. Dorent, R., et al. (2023). [CrossMoDA 2021 challenge: Benchmark of cross-modality domain adaptation techniques for vestibular schwannoma and cochlea segmentation](https://doi.org/10.1016/j.media.2022.102628). *Medical Image Analysis*, 83, 102628.
5. Isensee, F., et al. (2024). [nnU-Net Revisited: A Call for Rigorous Validation in 3D Medical Image Segmentation](https://doi.org/10.1007/978-3-031-72114-4_47). In *Medical Image Computing and Computer Assisted Intervention – MICCAI 2024* (pp. 488–498).